# Modelos de clasificación en Machine Learning con Python

**Mayo 2026 · Bloque III**

## Objetivos
- Entrenar clasificadores supervisados
- Evaluar accuracy, precision, recall, F1 y matriz de confusión
- Explicar la importancia de separar entrenamiento y prueba

## Preparación
Ejecuta la primera celda para cargar librerías. Si falta alguna librería, instálala desde el entorno con `pip install -r requirements.txt`.

## Carga y partición

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("/home/claude/datasets")
pd.set_option("display.max_columns", 50)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, RocCurveDisplay

df = pd.read_csv(DATA_DIR / "clientes_clasificacion.csv")
X = df.drop(columns=["abandono"])
y = df["abandono"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25, stratify=y, random_state=42)
df.head()

## Modelo base: Logistic Regression

In [ ]:
logit = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
logit.fit(X_train, y_train)
pred = logit.predict(X_test)
print(classification_report(y_test, pred, digits=3))

## Random Forest y matriz de confusión

In [ ]:
rf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight="balanced")
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
print(classification_report(y_test, pred_rf, digits=3))

ConfusionMatrixDisplay.from_estimator(rf, X_test, y_test)
plt.title("Matriz de confusión - Random Forest")
plt.show()

## Curva ROC

In [ ]:
RocCurveDisplay.from_estimator(logit, X_test, y_test, name="Logistic Regression")
RocCurveDisplay.from_estimator(rf, X_test, y_test, name="Random Forest")
plt.title("Curva ROC")
plt.show()

## Actividad entregable
1. Modifica el dataset o hiperparámetros.
2. Añade una breve interpretación de resultados.
3. Guarda el notebook ejecutado y exporta una versión HTML/PDF si se solicita.

## Interpretación de resultados

### Dataset
El dataset `clientes_clasificacion.csv` contiene **1 000 clientes** con 8 variables (edad, meses como cliente, número de productos, saldo medio, transacciones, contactos con soporte, si tiene tarjeta y score de satisfacción) y una variable objetivo binaria `abandono` (1 = sí abandona, 30% de los casos).

### Resultados

| Modelo | Accuracy | Precision (clase 1) | Recall (clase 1) | F1 (clase 1) |
|---|---|---|---|---|
| Regresión Logística | **90.8%** | 84.2% | **85.3%** | **84.8%** |
| Random Forest | 90.0% | **84.7%** | 81.3% | 83.0% |

La **Regresión Logística** supera ligeramente al Random Forest en este dataset porque las relaciones entre variables y abandono son aproximadamente lineales (se generaron así). En datasets reales con interacciones complejas, el Random Forest suele ser más robusto.

### Variable más importante
`meses_cliente` explica el **69 %** de la importancia en el Random Forest. Los clientes más nuevos tienen mayor riesgo de abandono, lo que tiene sentido de negocio.

### Conclusión
Ambos modelos son viables para un sistema de detección de churn. Se recomienda optimizar el **recall** (minimizar falsos negativos), ya que perder un cliente que iba a abandonar es más costoso que enviar un email de retención innecesario.
